# T06 + T07 — Cổng khả thi kỹ thuật trên Tesla T4

Notebook này chạy **cả hai task trong một phiên** để tiết kiệm quota GPU (30 giờ/tuần). Không chứa logic nào, chỉ clone, cài và gọi script.

**Notebook settings trước khi chạy:**

- Accelerator: **GPU T4 x2**
- Internet: **On**
- Data: attach dataset `unicorn1209/vihallulens` — T07 **bắt buộc** cần, vì phải chạy trên mẫu ISE-DSC01 dài nhất
- Add-ons → Secrets: `HF_TOKEN` (không bắt buộc)

Chạy hết từ trên xuống rồi copy output của **ô 5 và ô 6** dán vào PR.

In [ ]:
!git clone -q https://github.com/wsunicorn/vihallulens.git /kaggle/working/vihallulens
%cd /kaggle/working/vihallulens
!git log --oneline -1

In [ ]:
# Không cài lại torch: image Kaggle đã có bản dựng theo đúng CUDA của máy.
!pip install -q --no-deps -e .
!pip install -q -U bitsandbytes accelerate transformers

In [ ]:
import os

try:
    from kaggle_secrets import UserSecretsClient

    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if os.path.isdir("/kaggle/input/vihallulens") and not os.path.exists("data/raw"):
    os.makedirs("data", exist_ok=True)
    os.symlink("/kaggle/input/vihallulens", "data/raw")
print("data/raw ->", os.path.realpath("data/raw") if os.path.exists("data/raw") else "CHƯA GẮN")

In [ ]:
# Kiểm tra nhanh trên mô hình tí hon chạy CPU: hook có nhận được attn_weights không.
# Nếu ô này hỏng thì đừng chạy tiếp, vấn đề nằm ở phiên bản transformers.
!python scripts/probe_attention_hook.py --tiny

In [ ]:
# Ô 5 — T06. Copy output dán vào PR.
!python scripts/probe_load_model.py

In [ ]:
# Ô 6 — T07. Copy output dán vào PR.
!python scripts/probe_attention_hook.py

## Nếu ô 6 báo hết bộ nhớ

Đi theo bảng sáu nấc lùi ở mục 5 của `CLAUDE.md`, **theo thứ tự**, đừng nhảy cóc:

```python
# Nấc 1 — hạ ngân sách token. Đo ở T05: chỉ cắt thêm 1,09 % mẫu ISE-DSC01.
!python scripts/probe_attention_hook.py --max-context-tokens 2048

# Nấc 4 — lùi mô hình, cùng họ nên không đổi dòng code nào.
!python scripts/probe_attention_hook.py --model Qwen/Qwen2.5-3B-Instruct
```

Nấc 2 và nấc 3 cần sửa code, nấc 3 là đổi kiến trúc nên phải hỏi trước. Ghi lại nấc nào đã thử vào **Nhật ký chặn** cuối `TASKS.md`.